In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import StandardScaler



In [ ]:
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', 30)

# Load Raw Data

In [ ]:
df = pd.read_csv("spotify_tracks.csv")
df.head()

In [ ]:
df.shape

In [ ]:
df.isnull().sum()

# Remove Unnecessary Columns

In [ ]:
df1 = df.drop(['track_id','artwork_url','track_url'],axis = 'columns')
df1.shape

In [ ]:
df1.columns

# Select Audio Features for Recommendation

In [ ]:
features = ['acousticness', 'danceability','energy','instrumentalness','liveness','loudness','speechiness','valence','tempo']
X = df1[features]
X.head()

In [ ]:
X.describe()

# Identify Invalid Values
# -1 and -100000 represent invalid/missing values in the dataset
# These values can distort similarity calculations

In [ ]:
for col in X:
    print(col, (X[col] == -1).sum()) # Number of columns that have -1

In [ ]:
(X['loudness'] == -100000).sum() # Number of columns that have -100000 in Loudness

In [ ]:
X.isnull().sum()

In [ ]:
X.shape

# Identify Rows Containing invalid Values

In [ ]:
invalid_rows = (
    (X == -1).any(axis=1) |
    (X['loudness'] == -100000)
)

invalid_rows.sum()

# Remove the Invalid Rows from the Dataset

In [ ]:
X_clean = X[~invalid_rows].copy()
X_clean = X_clean.reset_index(drop=True)
X_clean.shape

In [ ]:
X_clean.describe()

In [ ]:
X_clean.head()

# Verify That Invalid Values Have Been Removed

In [ ]:
for x in X_clean:
    print(x, (X_clean[x] == -1).sum())

In [ ]:
(X_clean['loudness'] == -100000).sum()

# Danceability, Instrumentalness, Tempo Histograms

In [ ]:
plt.hist(X['danceability'], bins = 20, color = 'skyblue', edgecolor = 'black')
plt.xlabel("Danceability")
plt.ylabel("Number of songs")
plt.title("Danceability")

In [ ]:
plt.hist(X['instrumentalness'], bins = 20, color = 'skyblue', edgecolor = 'black')
plt.xlabel("Instrumentalness")
plt.ylabel("Number of songs")
plt.title("Instrumentalness")


In [ ]:
plt.hist(X['tempo'], bins = 20, color = 'skyblue', edgecolor = 'black')
plt.xlabel("Tempo")
plt.ylabel("Number of songs")
plt.title("Tempo")


# Correlation between music features


In [ ]:
corr = X_clean.corr()

sns.heatmap(corr, annot= True, cmap = 'coolwarm')
plt.title("Correlation between music features")
plt.show()

# Standardize Audio Features
# Standardization puts all features on a comparable scale

In [ ]:
scaler = StandardScaler()
scaled_data = scaler.fit_transform(X_clean)
scaled_X = pd.DataFrame(scaled_data,columns = features)
scaled_X.head()

In [ ]:
scaled_X.shape

# Create a Cleaned Dataset for Recommendations


In [ ]:
df_cleaned = df1[~invalid_rows].copy()
df_cleaned = df_cleaned.reset_index(drop=True)
df_cleaned.shape

In [ ]:
df_cleaned.head()

# Function to find a song in the dataset

In [ ]:
def find_song(song_name):

    # First, look for an exact match
    song = df_cleaned[
        df_cleaned['track_name'].str.lower() == song_name.lower()
    ]
    song = song.drop_duplicates(
    subset=['track_name', 'artist_name']
)

    # If no exact match, look for a partial match
    if song.empty:
        song = df_cleaned[
            df_cleaned['track_name'].str.lower().str.contains(
                song_name.lower(),
                na=False
            )
        ]

    # If no match at all
    if song.empty:
        print("\nSorry, that song is not in the dataset.")
        return None, None, None

		# If Multiple songs Found
    if len(song) > 1:
        print("\nMultiple Songs Found")

		# Display Choices
        for number, (index, row) in enumerate(song.iterrows(), start = 1):
            print(number, "-", row['track_name'], row['artist_name'])

		# Ask user to choose the song
        try:
            choice = int(input("\nChoose a song :"))
        except ValueError:
            print("\nPlease enter a number.")
            return None, None, None
      
		# Get selected song
        selected_song = song.iloc[choice - 1]
        song_index = selected_song.name
        actual_songname = selected_song['track_name']
        artist_name = selected_song['artist_name']
        return song_index, actual_songname, artist_name
				
    # If only one song was found
    song_index = song.index[0]
    actual_songname = song.iloc[0]['track_name']
    artist_name = song.iloc[0]['artist_name']
    
    return song_index, actual_songname, artist_name

In [ ]:
scaled_X.isna().sum()

# Song Recommendation Function


In [ ]:
def recommend_songs(song_name, song_language, number_of_songs):

    # Find the song
    song_index, actual_songname, artist_name = find_song(song_name)

    # Stop if the song was not found
    if song_index is None:
        return pd.DataFrame()

    # Print the selected song
    print("\nSelected song:")
    print("Song name : ", actual_songname)
    print("Artist name : ", artist_name)
    print("Language : ", song_language)

    # Calculate cosine similarity
    cosine = cosine_similarity(
        scaled_X.iloc[[song_index]],
        scaled_X
    )

    cosine = cosine.squeeze()

    # Add similarity scores to the dataset
    df_cleaned['Similarity'] = cosine

    # Filter by preferred language
    recommendations = df_cleaned[
        df_cleaned['language'].str.lower() == song_language.lower()
    ].copy()

    # Sort by similarity
    recommendations = recommendations.sort_values(
        by='Similarity',
        ascending=False
    )

    # Remove duplicate songs
    # Same song + same artist = duplicate
    recommendations = recommendations.drop_duplicates(
        subset=['track_name', 'artist_name']
    )
    # Don't Recommend the song itself
    recommendations = recommendations[recommendations['track_name'] != actual_songname]

    # Take requested number of songs
    recommendations = recommendations.head(number_of_songs)

    # Convert similarity to percentage
    recommendations['Similarity'] = (
        recommendations['Similarity'] * 100
    ).round(2)

    # Rename column
    recommendations = recommendations.rename(
        columns={'Similarity': 'Similarity (%)'}
    )

    # Select columns to display
    recommendations = recommendations[
        [
            'track_name',
            'artist_name',
            'language',
            'Similarity (%)'
        ]
    ]

    return recommendations

In [ ]:
while True:

    song_name = input("Enter a song: ").strip()

    song_language = input(
        "Enter your preferred language for recommendation : "
    ).strip()

    try:

        # Check if song is empty
        if not song_name:
            raise ValueError("Song name cannot be empty")

        # Check if language is empty
        if not song_language:
            raise ValueError("Song language cannot be empty")

        # Check if language exists
        if song_language.lower() not in df_cleaned['language'].str.lower().unique():
            raise ValueError("Language not found")

        # Ask for the number of recommendations
        number_of_songs = int(
            input(
                "Enter the number of recommendations you want : "
            )
        )

        # Check if number of songs exists
        if not number_of_songs:
            raise ValueError(
                "Number of Recommendations cannot be empty"
            )

        # If number is less than or equal to 0
        if number_of_songs <= 0:
            raise ValueError(
                "Number must be greater than 0"
            )

        # If number is more than 50
        if number_of_songs > 50:
            raise ValueError(
                "You can only request maximum of 50 recommendations"
            )

        # Get recommendations
        recommendations = recommend_songs(
            song_name,
            song_language,
            number_of_songs
        )

    except ValueError as e:

        print("Error :", e)

    else:

        if recommendations.empty:

            print("Sorry, no recommendations found.")

        else:

            print("\nRecommended Songs:\n")

            for number, (index, row) in enumerate(
                recommendations.iterrows(),
                start=1
            ):
                print(
                    number,
                    "-",
                    row['track_name'],
                    "-",
                    row['artist_name'],
                    "-",
                    row['language'],
                    "-",
                    row['Similarity (%)']
                )

    # Ask if user wants to search again
    while True:

        again = input(
            "\nWould you like to search again? (y/n)"
        ).strip().lower()

        if again == "y":
            break

        elif again == "n":
            break

        else:
            print("Please enter y or n")

    # Stop the main loop
    if again == "n":

        print("\nThank you for using music recommender!")

        break